# 05a - conservative feature ablation (U.S. training only)

**Purpose:** determine whether a small set of extra launch-time features improves the existing U.S. baseline. The original `05_modeling.ipynb` is kept intact. This notebook compares **the same estimator and hyperparameters** with and without extra features, under exactly the same five CV folds. No tuning, test evaluation, or changes to global notebooks occur here.

Extra candidates: `goal_vs_state` (training-fold state median), `goal_round_1000`, `duration_exact_30`, and `duration_exact_60`. These are hypothesis-driven experiments, not automatically approved features. The dataset still uses a provisional U.S. location proxy.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

HERE = Path.cwd()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents]
                     if (p / "utils" / "preprocessing.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run Jupyter from the repository root or a notebook subdirectory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from utils.preprocessing import drop_leaks, make_pipeline
from utils.us_launch_features import USLaunchFeatureBuilder

US = PROJECT_ROOT / "data" / "Processed-Data" / "us_only"
clean = pd.read_csv(US / "DSI_kickstarterscrape_dataset_cleaned.csv")
train_ids = pd.read_csv(US / "train_project_ids.csv")
rows = clean.set_index("project id", drop=False).loc[train_ids["project id"]].copy().reset_index(drop=True)
y_train = train_ids["target"].astype(int).reset_index(drop=True)
assert np.array_equal(rows["status"].eq("successful").astype(int), y_train)
X_train = drop_leaks(rows.drop(columns=["status"]))
assert not {"pledged", "backers", "funded percentage", "updates", "comments"}.intersection(X_train.columns)
print(f"Training projects (no test data loaded): {len(X_train):,}")


Training projects (no test data loaded): 26,945


In [2]:
# Preserve the exact baseline estimator settings from us_only/05.
SEED = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
FEATURE_POLICY = dict(include_timing=False, include_flags=False,
                      include_reward=False, exclude_renamed=True)
MODELS = {
    "logistic": (LogisticRegression(max_iter=2000), "onehot"),
    "hist_gradient_boosting": (
        HistGradientBoostingClassifier(max_iter=200, learning_rate=0.06, random_state=SEED),
        "target"
    ),
}
SCORING = {"roc_auc": "roc_auc", "accuracy": "accuracy", "f1": "f1"}
rows_out = []
for model_name, (model, strategy) in MODELS.items():
    for variant in ["original conservative", "additional safe candidates"]:
        pipeline = make_pipeline(clone(model), categorical_strategy=strategy, **FEATURE_POLICY)
        if variant == "additional safe candidates":
            # Refit both feature-builder medians and target encoding inside each CV fold.
            pipeline.set_params(features=USLaunchFeatureBuilder(**FEATURE_POLICY))
        scores = cross_validate(pipeline, X_train, y_train, cv=CV,
                                scoring=SCORING, n_jobs=1, error_score="raise")
        rows_out.append({
            "model": model_name,
            "variant": variant,
            **{metric: float(scores[f"test_{metric}"].mean()) for metric in SCORING},
            "auc_sd": float(scores["test_roc_auc"].std()),
            "fold_auc": scores["test_roc_auc"].copy(),
        })
results = pd.DataFrame(rows_out)
display(results.drop(columns=["fold_auc"]).round(4))


,model,variant,roc_auc,accuracy,f1,auc_sd
0,logistic,original conservative,0.7199,0.6684,0.7111,0.0055
1,logistic,additional safe candidates,0.7235,0.6700,0.7112,0.0066
2,hist_gradient_boosting,original conservative,0.7235,0.6684,0.7156,0.0082
3,hist_gradient_boosting,additional safe candidates,0.7232,0.6690,0.7162,0.0081


In [3]:
# Paired fold differences make tiny changes easy to see.
# They do not constitute a formal statistical significance test.
deltas = []
for model_name in MODELS:
    pair = results[results["model"].eq(model_name)].set_index("variant")
    original = pair.loc["original conservative", "fold_auc"]
    extended = pair.loc["additional safe candidates", "fold_auc"]
    d = np.asarray(extended) - np.asarray(original)
    deltas.append({
        "model": model_name,
        "mean_auc_delta": float(d.mean()),
        "fold_min": float(d.min()),
        "fold_max": float(d.max()),
        "positive_folds": int((d > 0).sum()),
    })
comparison = pd.DataFrame(deltas).set_index("model")
display(comparison.round(4))


,mean_auc_delta,fold_min,fold_max,positive_folds
model,,,,
logistic,0.0036,0.0013,0.0050,5
hist_gradient_boosting,-0.0003,-0.0014,0.0009,2


## Decision guidance for the team (before `06`)

- **Keep the original conservative baseline** if improvements are tiny, inconsistent across folds, or negative; these extras are optional.
- An exploratory improvement is **not** final evidence of predictive benefit; the CV is being used to decide whether candidates merit further work, and the official held-out test remains sealed.
- The goal- and duration-based fields are assumed to be available around launch; confirm their original snapshots and ensure the category/state reference statistics can be computed from historical training data at deployment.
- Discuss feature choices with Zasha; `06` performs tuning/model selection, then a single final test evaluation. Global results are preserved independently.
